# 14 · Load a dataset in batches
Level L2 · Part 2 · builds on lesson 13 (and lessons 07-12)

Needs: nothing for the main path. One `needs-download` cell re-embeds the recipes with MiniLM
(about 90 MB, cached once); otherwise the saved vectors in `data/` are used. The `needs-docker`
cells start one Qdrant server (`qdrant/qdrant:v1.19.2`, ports 6333 and 6334 free). When skipped,
the lesson prints the **recorded output** of a real run from `data/recorded/14.json`.

## Goal
By the end of this lesson you can load 2,000 recipes into Qdrant in batches of 256 with ids that
stay the same on every reload, and measure how much faster that is than one point at a time.

## The idea
Real data is a file of thousands of rows that gets reloaded every time it changes. Three habits make that fast and safe. **Batching**: send a few hundred points per
request instead of one, because every request to a server pays a fixed cost (a network **round
trip**) whatever its size. **Stable ids**: derive each point's id from something that never
changes about the recipe (its `slug`), so a reload overwrites the same points instead of mixing
them up. **Indexes first**: create payload indexes before the bulk load, so every point is
indexed as it arrives. It is like carrying shopping from the car: one bag per trip, not one apple
per trip.

## Picture

```mermaid
flowchart LR
    CSV["recipes_2000.csv<br/>2,000 rows"] --> ID["id = uuid5(slug)<br/>same id every load"]
    EMB["2,000 MiniLM vectors"] --> PTS
    ID --> PTS["2,000 points"]
    PTS --> B["8 batches<br/>7 x 256 + 1 x 208"]
    B -->|"8 requests<br/>instead of 2,000"| Q["collection 'recipes'<br/>payload indexes created first"]
```

## Step by step
### Step 1 · Two thousand recipes
`data/recipes_2000.csv` holds the course's 50 recipes plus 1,950 more made from templates
("Smoky chickpea tacos", "Herby salmon risotto", ...) by `data/make_recipes_2000.py`, so the file
is the same on every machine. Each row has a `slug`: a short, unique, lowercase name such as
`beef-stew`. There is no numeric id column; Step 3 explains why we don't want one.

In [1]:
import itertools, json, math, os, time, uuid, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", message="IProgress")     # quiet a progress-bar notice on import
from qdrant_client import QdrantClient, models

recipes = pd.read_csv("../data/recipes_2000.csv")
print(recipes.shape, "| columns:", list(recipes.columns))
print("vegetarian share:", round(recipes.vegetarian.mean(), 2), "| cuisines:", recipes.cuisine.nunique())
print(recipes.iloc[[7, 50, 1999]][["slug", "title", "cuisine", "minutes"]].to_string())

(2000, 6) | columns: ['slug', 'title', 'description', 'cuisine', 'vegetarian', 'minutes']
vegetarian share: 0.55 | cuisines: 14
                        slug                   title         cuisine  minutes
7                  beef-stew               Beef stew          French      180
50       spicy-chicken-curry     Spicy chicken curry          Indian       50
1999  herby-tempeh-flatbread  Herby tempeh flatbread  Middle Eastern       15


### Step 2 · One vector per recipe
Same model and text as lessons 05-13: MiniLM on `title + ". " + description`. Embedding 2,000
texts takes about 15 seconds on a laptop CPU. The next cell needs the model download; it saves
the vectors to `data/recipe_embeddings_minilm_2000.npz`, which is already in the repo, so skipping
it changes nothing below.

In [2]:
os.environ.setdefault("TQDM_DISABLE", "1")
os.environ.setdefault("HF_HUB_VERBOSITY", "error")
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
texts = (recipes.title + ". " + recipes.description).tolist()
fresh = model.encode_document(texts, batch_size=64)
np.savez_compressed("../data/recipe_embeddings_minilm_2000.npz", model="all-MiniLM-L6-v2",
                    slugs=recipes.slug.to_numpy(dtype=str), doc_emb=fresh)
print("embedded and saved:", fresh.shape, fresh.dtype)

⏭  Skipped: this cell needs [download]. Run it yourself, or re-run with TUT_ENABLE=download.


Load the saved vectors and check they line up with the CSV row by row (same slugs, same order).
A vector attached to the wrong recipe is the worst kind of bug: nothing fails, the search is just
wrong.

In [3]:
saved = np.load("../data/recipe_embeddings_minilm_2000.npz")
doc_emb, MODEL = saved["doc_emb"], str(saved["model"])
assert (saved["slugs"] == recipes.slug.to_numpy()).all(), "vectors and CSV rows are out of step"
print("model:", MODEL, "| vectors:", doc_emb.shape, doc_emb.dtype)

model: all-MiniLM-L6-v2 | vectors: (2000, 384) float32


### Step 3 · Ids that survive a reload
The tempting id is the row number. But the file is exported again tomorrow, and nothing promises
the same order. Say the next export comes sorted by title. With row numbers as ids, the same
ids now name different recipes:

In [4]:
tomorrow = recipes.sort_values("title", ignore_index=True)     # same recipes, new order
by_row_today = dict(enumerate(recipes.slug))
by_row_tomorrow = dict(enumerate(tomorrow.slug))
moved = sum(by_row_today[i] != by_row_tomorrow[i] for i in by_row_today)
print("id 7 today:", by_row_today[7], "| id 7 tomorrow:", by_row_tomorrow[7])
print("row-number ids that now point at a different recipe:", moved, "of", len(recipes))

id 7 today: beef-stew | id 7 tomorrow: chili-con-carne
row-number ids that now point at a different recipe: 1997 of 2000


Reloading tomorrow's file would overwrite almost every point with another recipe's data, and any
edit or click count stored against id 7 would now belong to a different dish. The fix is lesson
09's `uuid5`: a **deterministic id** computed from the recipe's own key. The same slug gives the
same UUID on any machine, in any order, forever.

In [5]:
RECIPE_NS = uuid.uuid5(uuid.NAMESPACE_URL, "https://example.com/recipes/")   # our own namespace

def recipe_id(slug):
    return str(uuid.uuid5(RECIPE_NS, slug))

ids_today = dict(zip(recipes.slug, recipes.slug.map(recipe_id)))
ids_tomorrow = dict(zip(tomorrow.slug, tomorrow.slug.map(recipe_id)))
print("beef-stew ->", recipe_id("beef-stew"))
print("ids that changed between the two exports:", sum(ids_today[s] != ids_tomorrow[s] for s in ids_today))
print("unique ids:", len(set(ids_today.values())))

beef-stew -> 89acc474-1f23-5810-8b94-54de9b7a876e
ids that changed between the two exports: 0
unique ids: 2000


Zero changed, and 2,000 different ids for 2,000 slugs. The id is now a property of the recipe,
not of its position in a file.

### Step 4 · Collection and payload indexes first
`make_collection` (re)creates the collection and then, **before any point arrives**, adds a
payload index (lesson 11) for each field we filter on: `cuisine`, `vegetarian`, `minutes`.
Qdrant's bulk-upload guide says to create them before uploading: the server then indexes every
point as it is written, and its search index can take the filters into account from the start
instead of being rebuilt later.

In [6]:
INDEXES = {"cuisine": models.PayloadSchemaType.KEYWORD,
           "vegetarian": models.PayloadSchemaType.BOOL,
           "minutes": models.PayloadSchemaType.INTEGER}

def make_collection(client, name="recipes"):
    if client.collection_exists(name):
        client.delete_collection(name)
    client.create_collection(name, vectors_config=models.VectorParams(
        size=doc_emb.shape[1], distance=models.Distance.COSINE))
    for field, kind in INDEXES.items():
        client.create_payload_index(name, field_name=field, field_schema=kind)
    return name

local = QdrantClient(":memory:")
print("created:", make_collection(local))

created: recipes


/var/tmp/ipykernel_39693/4170405147.py:11: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  client.create_payload_index(name, field_name=field, field_schema=kind)


Local mode prints a `UserWarning` that payload indexes have no effect there (lesson 11): it scans
every point anyway. The calls are still worth making, because the same function will run against
the server in Step 7.

### Step 5 · Points, then batches
One `PointStruct` per row, with the stable id and the payload (Python types from `itertuples`,
lesson 09):

In [7]:
def to_payload(row):
    return {"slug": row.slug, "title": row.title, "description": row.description,
            "cuisine": row.cuisine, "vegetarian": bool(row.vegetarian), "minutes": int(row.minutes)}

def to_points(df, emb):
    return [models.PointStruct(id=recipe_id(r.slug), vector=emb[i].tolist(), payload=to_payload(r))
            for i, r in enumerate(df.itertuples())]

points = to_points(recipes, doc_emb)
print(len(points), "points | first:", points[0].id, points[0].payload["title"])

2000 points | first: 21de3f0b-496e-5dca-9313-e9b7fb0a450b Tomato soup


`itertools.batched(items, n)` (Python 3.12+) cuts a list into chunks of `n`; the last one is
shorter. Each chunk is a **tuple**, and `upsert` wants a list, so we convert it (Common mistakes
shows what happens if you don't). `load` sends one `upsert` per batch and returns the seconds it
took.

In [8]:
sizes = [len(batch) for batch in itertools.batched(points, 256)]
print(len(sizes), "batches of 256:", sizes)

def load(client, points, batch_size, name="recipes"):
    start = time.perf_counter()
    for batch in itertools.batched(points, batch_size):
        client.upsert(name, points=list(batch))
    return time.perf_counter() - start

print("ready: load(client, points, batch_size)")

8 batches of 256: [256, 256, 256, 256, 256, 256, 256, 208]
ready: load(client, points, batch_size)


### Step 6 · Measure: one at a time vs batches, in local mode
**Throughput** = points written per second. We load all 2,000 points twice into a fresh
collection: once with `batch_size=1` (2,000 calls), once with 256 (8 calls).

In [9]:
def measure(client, batch_sizes):
    rows = {}
    for size in batch_sizes:
        make_collection(client)
        seconds = load(client, points, size)
        rows[f"upsert x {size}"] = (seconds, client.count("recipes").count)
    return rows

def show(rows):
    print(f"{'method':<22}{'calls':>6}{'seconds':>9}{'points/s':>10}{'points':>8}")
    for label, (seconds, count) in rows.items():
        calls = math.ceil(len(points) / int(label.split()[-1]))
        print(f"{label:<22}{calls:>6}{seconds:>9.2f}{count / seconds:>10,.0f}{count:>8}")

show(measure(local, [1, 256]))

method                 calls  seconds  points/s  points
upsert x 1              2000     0.86     2,317    2000
upsert x 256               8     0.71     2,814    2000


All 2,000 points arrive both ways, and batching helps only a little here (your seconds will
differ a bit from run to run). Local mode runs **inside your Python process**: a call costs a
function call, not a network trip, so there is little to save. The cost batching removes is the
round trip to a server. Time to measure that.

### Step 7 · Measure on a real server
The server cells only run with Docker (`infra/compose/qdrant-single.yml`, as in lesson 13). Their
results go into `server_log`; when they are skipped, `from_server` reads the recorded output of
a real run instead.

In [10]:
RECORDED = "../data/recorded/14.json"
server_log = {}                                          # filled by the needs-docker cells

def from_server(key):
    if key in server_log:
        return server_log[key], "live"
    with open(RECORDED) as f:
        return json.load(f)[key], "recorded output"

from pathlib import Path
print("recorded keys:", sorted(json.loads(Path(RECORDED).read_text())) if os.path.exists(RECORDED) else "none")

recorded keys: ['payload_schema', 'timings', 'upload_points', 'version']


Start the server and poll `/readyz` until it answers (lesson 13's `wait_ready`):

In [11]:
import socket, subprocess, urllib.request
QDRANT_URL = "http://localhost:6333"
COMPOSE = ["docker", "compose", "-f", "../infra/compose/qdrant-single.yml"]

def wait_ready(url, timeout=60):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            return urllib.request.urlopen(f"{url}/readyz", timeout=2).read().decode()
        except OSError:
            time.sleep(0.5)
    raise TimeoutError(f"{url} not ready after {timeout}s")

if socket.socket().connect_ex(("localhost", 6333)) == 0:
    raise RuntimeError("port 6333 is already in use; stop the other Qdrant first")
subprocess.run(COMPOSE + ["up", "-d"], check=True, capture_output=True)
print("readyz:", wait_ready(QDRANT_URL))
server = QdrantClient(url=QDRANT_URL)
server_log["version"] = json.load(urllib.request.urlopen(QDRANT_URL))["version"]
print("server version:", server_log["version"])

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


Now the same `measure`, against the server, for five batch sizes. One point per call means 2,000
HTTP requests, so this cell takes about 15 seconds. Each `upsert` waits until the server has
applied the batch (`wait=True`, the default, lesson 12), so the clock measures real writes.

In [12]:
server_rows = measure(server, [1, 16, 64, 256, 1024])
server_log["timings"] = {label: [round(s, 3), n] for label, (s, n) in server_rows.items()}
show(server_rows)

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


Here is the table (live, or the recorded run if Docker was skipped):

In [13]:
timings, source = from_server("timings")
print(f"server timings ({source}):")
show({label: tuple(value) for label, value in timings.items()})
speedup = timings["upsert x 1"][0] / timings["upsert x 256"][0]
print(f"batches of 256 vs one at a time: {speedup:.1f}x faster")

server timings (recorded output):
method                 calls  seconds  points/s  points
upsert x 1              2000     8.71       230    2000
upsert x 16              125     1.51     1,327    2000
upsert x 64               32     1.02     1,965    2000
upsert x 256               8     0.90     2,217    2000
upsert x 1024              2     0.86     2,312    2000
batches of 256 vs one at a time: 9.7x faster


On a server, batching matters a lot: going from one point per request to 256 cut the load time
by roughly ten times in our runs. Most of the gain is already there at 64; beyond 256 the
numbers barely move (and wobble from run to run), because the round trips are no longer the
bottleneck. Qdrant's guide recommends **64-256 points per batch**: smaller wastes round trips,
bigger uses more server memory per request and makes a failed batch more expensive to retry.

### Step 8 · `upload_points`: batching done for you
qdrant-client has a helper for exactly this loop. `upload_points` takes any iterable of points,
cuts it into `batch_size` chunks, and retries a failed batch. Its defaults: `batch_size=64`,
`max_retries=3`, `parallel=1`, and, unlike `upsert`, `wait=False`: it does **not** wait for the
server to apply the last batch. Pass `wait=True` when the next line reads the data. `parallel=2` or more starts
worker processes, which only pays off on loads far bigger than ours.

In [14]:
make_collection(server)
start = time.perf_counter()
server.upload_points("recipes", points, batch_size=256, wait=True)
seconds = time.perf_counter() - start
server_log["upload_points"] = [round(seconds, 3), server.count("recipes").count]
print(f"upload_points: {seconds:.2f} s, {server_log['upload_points'][1]} points")

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


Compared with our own loop at 256:

In [15]:
(up_seconds, up_count), source = from_server("upload_points")
print(f"{'our loop, 256 per upsert':<26}{timings['upsert x 256'][0]:>6.2f} s")
print(f"{'upload_points, 256':<26}{up_seconds:>6.2f} s   ({up_count} points, {source})")

our loop, 256 per upsert    0.90 s
upload_points, 256          0.39 s   (2000 points, recorded output)


`upload_points` was faster again. Same batch size, same server, so the
gap is client-side: `upsert` checks every point it is given for local-inference objects on every
call, while `upload_points` checks only the first point and then streams batches straight to the
server. Use it for bulk loads; keep `upsert`
for small writes where you want the result straight away.

The indexes from Step 4 now cover every point. The server reports each index and how many
points it holds:

In [16]:
schema = server.get_collection("recipes").payload_schema
server_log["payload_schema"] = {f: [str(i.data_type.value), i.points] for f, i in schema.items()}
print(server_log["payload_schema"])

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


Side by side with local mode, which keeps no payload indexes:

In [17]:
server_schema, source = from_server("payload_schema")
print("local mode:", local.get_collection("recipes").payload_schema)
for field, (kind, count) in server_schema.items():
    print(f"server: {field:<11}{kind:<8}{count} points   ({source})")

local mode: {}
server: cuisine    keyword 2000 points   (recorded output)
server: vegetarian bool    2000 points   (recorded output)
server: minutes    integer 2000 points   (recorded output)


### Step 9 · Reload safely
Because ids come from slugs, loading the same file again is **idempotent** (lesson 09): it
overwrites the same 2,000 points instead of adding more. Locally, `upload_points` ignores
`batch_size` and writes everything in one go (it behaves like `wait=True`), so it is also the
simplest way to fill a local collection.

In [18]:
make_collection(local)
local.upload_points("recipes", points)
local.upload_points("recipes", points)                  # the same file, loaded twice
print("points after two loads:", local.count("recipes").count)

points after two loads: 2000


A plain reload has one danger: it **replaces** whole points (lesson 09), so any edit made in the
database since the last load is lost. Say the chef shortens the beef stew to 150 minutes directly
in the database, and then the nightly job reloads the file:

In [19]:
stew = [recipe_id("beef-stew")]
local.set_payload("recipes", payload={"minutes": 150}, points=stew)
print("after the chef's edit:", local.retrieve("recipes", ids=stew)[0].payload["minutes"])
local.upload_points("recipes", points)
print("after a plain reload: ", local.retrieve("recipes", ids=stew)[0].payload["minutes"])

after the chef's edit: 150


after a plain reload:  180


The edit is gone. When the database, not the file, owns later edits, load with
`update_mode=INSERT_ONLY`. `update_mode` is the setting from lesson 12, where you used
`UPDATE_ONLY`. Its other value, `INSERT_ONLY`, does the opposite: it writes points whose id is
new and silently skips ids that already exist (Qdrant docs: "If a point with the same ID exists,
the operation is ignored"). Start
from yesterday's export (the first 1,900 recipes), make the edit, then load today's full file:

In [20]:
make_collection(local)
local.upload_points("recipes", points[:1900])             # yesterday: 1,900 recipes
local.set_payload("recipes", payload={"minutes": 150}, points=stew)
local.upload_points("recipes", points, update_mode=models.UpdateMode.INSERT_ONLY)   # today
print("points:", local.count("recipes").count)
print("beef stew minutes:", local.retrieve("recipes", ids=stew)[0].payload["minutes"])

points: 2000
beef stew minutes: 150


The 100 new recipes arrived and the edit survived. Pick the mode on purpose: plain upsert when
the file is the source of truth, `INSERT_ONLY` when edits in the database must win.

### Step 10 · Check the load with a search
A load is done when it is checked, not when it returns: the count matches the file, and a known
query still gives sensible answers. "a hot curry" (from the query cache), vegetarian and at most
30 minutes:

In [21]:
cache = np.load("../data/query_embeddings_minilm.npz")
curry_query = cache["query_emb"][10].tolist()
quick_veg = models.Filter(must=[
    models.FieldCondition(key="vegetarian", match=models.MatchValue(value=True)),
    models.FieldCondition(key="minutes", range=models.Range(lte=30))])
assert local.count("recipes").count == len(recipes)
hits = local.query_points("recipes", query=curry_query, query_filter=quick_veg, limit=5).points
print("query:", str(cache["queries"][10]))
for h in hits:
    print(f"  {h.score:.3f}  {h.payload['title']:<32} {h.payload['minutes']:>3} min")

query: a hot curry
  0.560  Spicy tofu green curry            25 min
  0.536  Smoky tofu green curry            30 min
  0.503  Herby tempeh green curry          30 min
  0.502  Creamy tempeh curry               30 min
  0.482  Spicy spinach green curry         25 min


Five curries, all vegetarian and quick, all from the 1,950 new recipes. Among the original 50
the only vegetarian curry-like dish is Red lentil dal, so a bigger menu gives the search real
choices. The scores sit close together (0.48-0.56) because the generated descriptions share
templates; lesson 15 builds the full recipe finder on this collection.

## What just happened
- Step 3: with row-number ids, re-sorting the file made almost every id point at another recipe;
  with `uuid5(slug)` ids, 0 of 2,000 changed.
- Step 4: payload indexes were created before the first point; the server reported all 2,000
  points in each of the three indexes after the load.
- Steps 6-8: in local mode batching barely changed the speed (no network); on the server,
  batches of 256 were about ten times faster; `upload_points` was faster again at the same batch
  size.
- Step 9: reloading the same file left 2,000 points; `INSERT_ONLY` added the 100 new recipes and
  kept the chef's edit, which a plain reload had overwritten.

## Common mistakes

**1. Passing a batch from `itertools.batched` straight to `upsert`.** The batches are tuples, and
qdrant-client accepts a list of points or a `models.Batch`, not a tuple.

In [22]:
try:
    make_collection(local)
    for batch in itertools.batched(points, 256):
        local.upsert("recipes", points=batch)
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

ValueError: Unsupported type: <class 'tuple'>


It prints `ValueError: Unsupported type: <class 'tuple'>` on the first batch, so nothing was
written. (A server client fails differently, with `ResponseHandlingException: object of type
'PointStruct' has no len()`.) Fix: `points=list(batch)`, as `load` does, or let `upload_points`
do the batching.

In [23]:
make_collection(local)
seconds = load(local, points, 256)
print("loaded:", local.count("recipes").count, "points")

loaded: 2000 points


**2. Giving `uuid5` a plain string as its namespace.** The first argument must itself be a UUID.

In [24]:
try:
    uuid.uuid5("recipes", "beef-stew")
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

AttributeError: 'str' object has no attribute 'bytes'


It prints `AttributeError: 'str' object has no attribute 'bytes'`. Fix: use one of Python's
built-in namespaces (`uuid.NAMESPACE_URL`) or, as in Step 3, make your own once with
`uuid5(NAMESPACE_URL, "https://example.com/recipes/")` and keep it fixed forever: a different
namespace gives every recipe a different id.

In [25]:
print(uuid.uuid5(uuid.NAMESPACE_URL, "beef-stew"))      # built-in namespace works
print(recipe_id("beef-stew"))                           # our fixed namespace
print("same id as Step 3:", recipe_id("beef-stew") == "89acc474-1f23-5810-8b94-54de9b7a876e")

d0c9932c-2510-569c-85dd-2ec454c719bb
89acc474-1f23-5810-8b94-54de9b7a876e
same id as Step 3: True


Both calls work, but the two namespaces give `beef-stew` two different ids. That is why the
namespace must never change: switching it would give every recipe a new id on the next reload.

## Try it
1. **A nightly job that reports what it added.** Write `nightly_load(client, df, emb)` that loads
   with `INSERT_ONLY` and returns how many points were new (count after minus count before). On a
   fresh collection, call it with the first 1,500 recipes, then with all 2,000, then with all
   2,000 again. Expect 1,500, 500, 0.
2. **Columns instead of points.** `upload_collection(name, vectors=..., payload=..., ids=...,
   batch_size=256)` takes the vectors as one NumPy array and the payloads and ids as lists. Load
   the 2,000 recipes with it into a fresh collection, then check that the count is 2,000 and that
   `retrieve` on `recipe_id("ramen")` returns Ramen.

In [26]:
# your code here

Clean up: close the local client, then (with Docker) save what the server printed (this refreshes
the recorded output used when Docker is skipped) and stop the container. `down -v` also deletes
the server's data volume.

In [27]:
local.close()
print("local client closed")

local client closed


The Docker part of the clean-up:

In [28]:
server.close()
with open(RECORDED, "w") as f:
    json.dump(server_log, f, indent=1)
subprocess.run(COMPOSE + ["down", "-v"], check=True, capture_output=True)
print("saved", sorted(server_log), "to", RECORDED, "| server stopped")

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


## Key terms
- **bulk load** — writing a whole dataset in one job, with batches and indexes prepared first.
- **batch upsert** — one `upsert` call that writes many points (a list or a `models.Batch`).
- **batch size** — points per request; Qdrant recommends 64-256.
- **round trip** — one request to a server and its reply; a fixed cost that batching shares.
- **throughput** — work done per second; here, points written per second.
- **deterministic id** — an id computed from the record's permanent key (`uuid5(namespace,
  slug)`), so the same record gets the same id on every load.
- **upload_points** — qdrant-client helper that batches an iterable of points, retries failed
  batches, and by default does not wait for the last write (`wait=False`).
- **insert only** — `update_mode=INSERT_ONLY`: write new ids, leave existing points untouched.